# Helios 01 — Vector Engine: building footprints to vector PMTiles

![Overture buildings to vector PMTiles](https://raw.githubusercontent.com/databrickslabs/geobrix/main/resources/images/helios-01.png)

**Solar site-selection, step 1: the candidate surfaces.** Every rooftop in San
Francisco is a potential solar surface. This notebook pulls **Overture Maps building
footprints** for the city, encodes them into **Mapbox Vector Tiles** with
`gbx_st_asmvt`, pyramids them across zoom levels with `gbx_st_asmvt_pyramid`, folds
the whole pyramid into one **PMTiles** archive with `gbx_pmtiles_agg`, and views it
inline with `plot_pmtiles`. The result is a single self-contained vector basemap of
every candidate roof — the geometry layer the later notebooks score for solar yield.

Along the way we compose with **Databricks-native** `ST_*` and `H3` functions — GeoBrix
tiling is an on-ramp into the native spatial engine, not a replacement for it — to
quantify roof area and bin roofs into an H3 roof-density surface.

> Runs on the **lightweight tier (Serverless)** by default. See `config_nb` for the
> heavyweight switch.

In [ ]:
%run ./config_nb

In [ ]:
# Flip to True to fully rebuild this notebook's tables / re-download / re-tile.
FORCE_REBUILD = False

## 1. Discover Overture building assets for the SF AOI

`OvertureClient.discover` traverses Overture's static STAC catalog and returns one
row per GeoParquet asset that intersects our bbox — metadata only, on the driver.
We narrow to the `buildings` theme; `themes=None` would select every theme.

In [ ]:
# -- Offline guard: discover hits the network (stac.overturemaps.org). In Docker /
#    CI without network access we skip to a synthetic fallback so that the downstream
#    tiling + PMTiles cells can still execute and be validated.
import os

_OFFLINE = os.environ.get("GBX_HELIOS_OFFLINE", "").lower() in ("1", "true", "yes")

if not _OFFLINE:
    try:
        assets = overture.discover(SF_CITY_BBOX, themes=["buildings"])
        display(assets)        # theme, type, href, asset_bbox, release
        print(f"... {assets.count()} intersecting building assets")
    except Exception as _e:
        print(f"... discover failed ({type(_e).__name__}: {_e}); switching to offline mode")
        _OFFLINE = True

if _OFFLINE:
    print("... OFFLINE mode: Overture network unavailable; using synthetic SF buildings fallback")
    assets = None

## 2. Download the AOI subset to a Volume (+ catalog it in Delta)

`OvertureClient.download` reads only the AOI rows (bbox predicate pushdown) and
writes them to the Volume **distributed across workers**, returning a metadata
DataFrame and UPSERTing it into the `overture_buildings_meta` Delta table (idempotent
MERGE keyed by `theme, type, source`). On Serverless the download fans out via
`repartition(N, col)` — no cluster knobs to tune.

In [ ]:
OVERTURE_DIR = f"{HELIOS_DIR}/overture"

if not _OFFLINE:
    meta = overture.download(
        assets, OVERTURE_DIR,
        table="overture_buildings_meta",
        validate=True,
        partitions=64,                 # hash fan-out; Serverless-safe parallelism
    )
    display(meta.select("theme", "type", "source", "out_file_sz", "is_out_file_valid"))
else:
    meta = None
    print("... OFFLINE mode: skipping download")

## 3. Load the building geometries

`OvertureClient.read` loads the downloaded GeoParquet back into Spark, re-applying
the bbox AOI filter. We keep the building polygon geometry and a stable id.

In offline / Docker mode we substitute a small synthetic dataset of SF-area building
polygons (WKB-encoded) so the tiling and PMTiles cells can still execute.

In [ ]:
from pyspark.sql.types import StringType, BinaryType, StructType, StructField

if not _OFFLINE:
    buildings = (
        overture.read("overture_buildings_meta", theme="buildings", type="building", bbox=SF_CITY_BBOX)
                .select(F.col("id").alias("feature_id"), F.col("geometry"))
    )
    print(f"... {buildings.count():,} building footprints in the AOI")
    display(buildings.limit(5))
else:
    # Synthetic SF-area building footprints as WKB BINARY (EPSG:4326 polygons).
    # These are representative downtown + mission district parcels for offline validation.
    from shapely.geometry import box as _shapely_box
    from shapely import wkb as _wkb

    _sf_parcels = [
        # (id, minx, miny, maxx, maxy)  — small building footprints in SF
        ("b001", -122.4194, 37.7749, -122.4180, 37.7758),
        ("b002", -122.4200, 37.7760, -122.4185, 37.7770),
        ("b003", -122.4170, 37.7740, -122.4155, 37.7752),
        ("b004", -122.4220, 37.7780, -122.4210, 37.7790),
        ("b005", -122.4140, 37.7720, -122.4125, 37.7732),
        ("b006", -122.4160, 37.7800, -122.4148, 37.7812),
        ("b007", -122.4090, 37.7710, -122.4075, 37.7722),
        ("b008", -122.4230, 37.7770, -122.4218, 37.7780),
        ("b009", -122.4050, 37.7690, -122.4040, 37.7700),
        ("b010", -122.4100, 37.7810, -122.4088, 37.7820),
        ("b011", -122.4250, 37.7750, -122.4238, 37.7760),
        ("b012", -122.4080, 37.7760, -122.4068, 37.7770),
    ]

    _schema = StructType([
        StructField("feature_id", StringType(), False),
        StructField("geometry", BinaryType(), True),
    ])
    _rows = [
        (_id, bytes(_wkb.dumps(_shapely_box(minx, miny, maxx, maxy))))
        for _id, minx, miny, maxx, maxy in _sf_parcels
    ]
    buildings = spark.createDataFrame(_rows, schema=_schema)
    print(f"... {buildings.count()} synthetic SF building footprints (offline mode)")
    display(buildings.limit(5))

## 3b. Quantify candidate roof space with Databricks-native ST + H3

Before tiling, we use **Databricks-native** spatial functions on the same footprints —
GeoBrix tiling composes directly with the native engine. Native `st_area` /
`st_centroid` (over `st_geomfromwkb`) give each roof's **available area** and a point
to index; native `h3_longlatash3` bins those centroids into H3 cells so we can
aggregate **roof density and total roof area per cell** — a coarse "where are the
candidate solar surfaces concentrated?" view that complements the per-building tiles.

> **Requires Databricks Runtime with Photon or Databricks SQL (Pro/Serverless).** The
> cells below check for `st_geomfromwkb` at runtime and skip with a note if the native
> engine is unavailable (e.g. the plain-Spark Docker container). The tiling cells are
> unaffected.

In [ ]:
# Native ST: parse WKB -> GEOMETRY, then area (planar in the layer CRS) + centroid.
# st_geomfromwkb bridges GeoBrix WKB output to native Databricks ST.
# st_area here is planar in EPSG:4326 (square degrees — useful for relative
# comparison; for true m² work in a projected CRS or use GEOGRAPHY/st_area semantics
# per the Databricks ST reference).
_native_st_available = False
roofs = None
try:
    roofs = buildings.selectExpr(
        "feature_id",
        "geometry",
        "st_area(st_geomfromwkb(geometry)) AS roof_area_deg2",
        "st_x(st_centroid(st_geomfromwkb(geometry))) AS lon",
        "st_y(st_centroid(st_geomfromwkb(geometry))) AS lat",
    )
    display(roofs.orderBy(F.col("roof_area_deg2").desc()).limit(5))
    _native_st_available = True
except Exception as _e:
    print(f"... native ST functions unavailable ({type(_e).__name__}); skipping roof-area cell")
    print("    (st_geomfromwkb / st_area / st_centroid require Databricks Runtime with Photon or SQL Pro/Serverless)")

In [ ]:
# Native H3: index each roof centroid to an H3 cell (res 11 ~ city-block scale) and
# aggregate roof count + total area per cell -> a roof-density surface.
# h3_longlatash3(lon, lat, resolution) is the Databricks-native point->H3 built-in
# (returns BIGINT cell index; use h3_longlatash3string for the string encoding).
H3_RES = 11

if _native_st_available and roofs is not None:
    try:
        roof_density = (
            roofs.selectExpr("*", f"h3_longlatash3(lon, lat, {H3_RES}) AS h3_cell")
                 .groupBy("h3_cell")
                 .agg(F.count("*").alias("n_roofs"),
                      F.sum("roof_area_deg2").alias("total_roof_area_deg2"))
        )
        display(roof_density.orderBy(F.col("total_roof_area_deg2").desc()).limit(10))
    except Exception as _e:
        print(f"... h3_longlatash3 unavailable ({type(_e).__name__}); skipping H3 density cell")
        print("    (h3_longlatash3 requires Databricks Runtime with H3 built-ins)")
else:
    print("... skipping H3 roof-density (native ST unavailable or roofs DataFrame not built)")

## 4. Encode + pyramid to vector tiles

`gbx_st_asmvt_pyramid` is a table-valued function (UDTF): for each feature it emits
one `(z, x, y, mvt_bytes)` row per zoom level in the requested range, binning the
geometry into the web-mercator tile grid and encoding tile-local MVT. We pick a
city-scale zoom range (z12–z16). Attributes ride along natively (here just
`feature_id`).

In [ ]:
buildings.createOrReplaceTempView("sf_buildings")

# gbx_st_asmvt_pyramid (light tier — Python UDTF) signature:
#   gbx_st_asmvt_pyramid(geom_wkb, attrs, min_z, max_z [, layer_name [, extent]])
# Output columns (flat): z INTEGER, x INTEGER, y INTEGER, mvt_bytes BINARY
# Each row is one feature's contribution to one tile — multiple features that
# fall in the same (z,x,y) appear as separate rows; gbx_pmtiles_agg packs them
# (first-write-wins per tile-id when (z,x,y) duplicates occur).
mvt = spark.sql("""
    SELECT t.z, t.x, t.y, t.mvt_bytes
    FROM sf_buildings,
         LATERAL gbx_st_asmvt_pyramid(
             geometry,
             named_struct('feature_id', feature_id),
             12, 16,
             'buildings'
         ) AS t
""")

sf_mvt = finalize_delta(mvt, "sf_buildings_mvt_tiles")
print(f"... {sf_mvt.count():,} (z,x,y) MVT rows across z12-z16")

## 5. Fold the tile pyramid into one PMTiles archive

`gbx_pmtiles_agg` is a grouped aggregate that folds a set of `(bytes, z, x, y)` tiles
into a single PMTiles v3 archive (BINARY). We aggregate the whole pyramid into one
archive and write it to the Volume.

In [ ]:
# gbx_pmtiles_agg signature: gbx_pmtiles_agg(bytes, z, x, y [, metadata_json])
# Folds all rows into one PMTiles v3 archive; duplicate (z,x,y) keys are
# deduplicated (first-write wins), so the LATERAL fan-out rows land correctly.
archive_row = (
    sf_mvt.groupBy(F.lit(1).alias("_g"))
          .agg(F.expr("gbx_pmtiles_agg(mvt_bytes, z, x, y)").alias("archive"))
          .select("archive")
          .collect()[0]
)
TILES_DIR = f"{HELIOS_DIR}/tiles"
dbutils.fs.mkdirs(TILES_DIR)
PMTILES_PATH = f"{TILES_DIR}/sf_buildings.pmtiles"
# FUSE-safe sequential write from the driver (single archive, bytes already in memory)
with open(PMTILES_PATH, "wb") as f:
    f.write(archive_row["archive"])
print(f"... wrote {PMTILES_PATH} ({os.path.getsize(PMTILES_PATH):,} bytes)")

## 6. View the vector PMTiles inline

`show_pmtiles` prints the `pmtiles_info` header, then renders through the
`INTERACTIVE_PLOTS` toggle (set in `config_nb`): the default **False** produces a
fast static image that renders on GitHub and the docs site; set `INTERACTIVE_PLOTS =
True` for an interactive MapLibre layer (streamed in-browser, no tile server).
`plot_pmtiles` auto-detects the vector archive in either mode.

In [ ]:
show_pmtiles(PMTILES_PATH)

## What we built

- `overture_buildings_meta` (Delta) — the queryable asset catalog of downloaded GeoParquet.
- `sf_buildings_mvt_tiles` (Delta) — one row per `(z, x, y)` vector tile.
- `sf_buildings.pmtiles` (Volume) — a self-contained vector basemap of every candidate roof.
- A native **ST roof-area** table and an **H3 roof-density** aggregation — showing the
  tiles compose directly with Databricks-native spatial.

Next: **notebook 02** drapes NAIP aerial imagery behind these footprints as a visual basemap.